# Data Pipeline to merge, clean, and train model

In [2]:
#Importing libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import streamlit as st

In [3]:
#reading in employee data files
HR = 'Employee.csv'
HR_df = pd.read_csv(HR)

#reading in Performance data file
Perf = 'PerformanceRating.csv'
Perf_df = pd.read_csv(Perf)

In [4]:
HR_df.rename(columns={'DistanceFromHome (KM)':'DistanceFromHome_KM'}, inplace=True)

In [5]:
#Ensure the date column is in datetime format
Perf_df['ReviewDate'] = pd.to_datetime(Perf_df['ReviewDate'])

#Sort by date ascending, then keep the last entry for each item
Per_latest_df = Perf_df.sort_values('ReviewDate').reset_index().drop_duplicates(subset=['EmployeeID'], keep='last')

In [6]:
#Merge data on EmployeeID and keeping only rows that have matching EmployeeIDs
merged_df = pd.merge(Per_latest_df, HR_df, on='EmployeeID', how='inner')
pd.set_option('display.max_columns', None)
merged_df.sample(5)

,index,PerformanceID,EmployeeID,ReviewDate,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome_KM,State,Ethnicity,Education,EducationField,JobRole,MaritalStatus,Salary,StockOptionLevel,OverTime,HireDate,Attrition,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager
204,5154,PR5634,C330-DAF4,2022-01-14,4,2,4,3,0,4,5,4,Lonni,Lelievre,Female,34,Some Travel,Sales,16,NY,White,2,Economics,Sales Executive,Divorced,100710,1,Yes,2012-05-20,No,10,10,10,2
359,5329,PR5792,0322-D46B,2022-03-02,3,4,5,2,2,2,4,3,Nikolas,Leslie,Male,32,Some Travel,Sales,39,CA,Black or African American,4,Marketing,Sales Executive,Married,105748,2,No,2012-03-28,No,10,0,1,8
687,5686,PR6113,B3AF-7E58,2022-05-31,3,2,2,3,0,4,3,2,Elvira,Ianelli,Female,45,Some Travel,Human Resources,34,NY,Black or African American,2,Human Resources,Recruiter,Divorced,54132,1,Yes,2012-01-20,No,10,10,10,10
797,5816,PR6230,90EB-28D4,2022-07-08,3,2,3,3,1,3,3,2,Veronique,Tremelling,Female,24,Frequent Traveller,Sales,29,CA,Black or African American,1,Economics,Sales Representative,Single,26608,0,No,2017-09-12,No,5,0,3,1
837,5853,PR6264,F83C-CA1D,2022-07-18,4,4,5,3,2,2,3,3,Durante,Heap,Male,21,Some Travel,Technology,16,CA,White,1,Information Systems,Software Engineer,Single,22515,0,Yes,2019-12-05,Yes,1,0,0,0


In [7]:
#Cleaning function to handle yes/no columns and unwanted columns

def clean_df(df, yes_no_cols, cols_to_drop=None):
    
    """
    Cleans data file to be used for machine learning by mapping yes/no to 1 and 0, 
    creating dummy variables, and converting time to datetime.
    """
    
    # Create a copy of the dataframe
    cleaned_df = df.copy()

    # Strip whitespace from every string cell in the entire DataFrame
    # Safely ignores integers, floats, and NaNs
    cleaned_df = cleaned_df.map(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Remove unwanted columns
    if cols_to_drop:
        cleaned_df = cleaned_df.drop(columns=cols_to_drop, errors='ignore')
    
    # Convert Yes and No columns to 1 and 0
    if yes_no_cols:
        # Standardize strings to lowercase and strip whitespaces
        for col in yes_no_cols:
            cleaned_df[col] = cleaned_df[col].astype(str).str.lower().str.strip()
            # Map values (unknown/other values will become NaN)
            cleaned_df[col] = cleaned_df[col].replace({'yes': 1, 'y': 1, 'no': 0, 'n': 0}).astype(int)
        
    return cleaned_df

In [8]:
#Run merged dataframe through cleaning function
yes_no = ['StockOptionLevel','Attrition','OverTime']
drop_list = ['EmployeeID','FirstName','LastName','PerformanceID','ReviewDate','index']

merged_cleaned = clean_df(
    df=merged_df, 
    yes_no_cols=yes_no,  
    cols_to_drop=drop_list
)

merged_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1280 entries, 0 to 1279
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   EnvironmentSatisfaction          1280 non-null   int64 
 1   JobSatisfaction                  1280 non-null   int64 
 2   RelationshipSatisfaction         1280 non-null   int64 
 3   TrainingOpportunitiesWithinYear  1280 non-null   int64 
 4   TrainingOpportunitiesTaken       1280 non-null   int64 
 5   WorkLifeBalance                  1280 non-null   int64 
 6   SelfRating                       1280 non-null   int64 
 7   ManagerRating                    1280 non-null   int64 
 8   Gender                           1280 non-null   object
 9   Age                              1280 non-null   int64 
 10  BusinessTravel                   1280 non-null   object
 11  Department                       1280 non-null   object
 12  DistanceFromHome_KM              1

/var/folders/p4/vxx4n3kn4q1grbl878lw90k80000gp/T/ipykernel_93561/4289377345.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_df[col] = cleaned_df[col].replace({'yes': 1, 'y': 1, 'no': 0, 'n': 0}).astype(int)
/var/folders/p4/vxx4n3kn4q1grbl878lw90k80000gp/T/ipykernel_93561/4289377345.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_df[col] = cleaned_df[col].replace({'yes': 1, 'y': 1, 'no': 0, 'n': 0}).astype(int)


In [11]:
#Defining Features(X) and Target variables(y)

X = merged_cleaned.drop(columns=['Attrition','HireDate'])
y = merged_cleaned['Attrition']

#Cleaning function to handle yes/no columns and unwanted columns

def clean_df(df, yes_no_cols, cols_to_drop=None):
    
    """
    Cleans data file to be used for machine learning by mapping yes/no to 1 and 0, 
    creating dummy variables, and converting time to datetime.
    """
    
    # Create a copy of the dataframe
    cleaned_df = df.copy()

    # Strip whitespace from every string cell in the entire DataFrame
    # Safely ignores integers, floats, and NaNs
    cleaned_df = cleaned_df.map(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Remove unwanted columns
    if cols_to_drop:
        cleaned_df = cleaned_df.drop(columns=cols_to_drop, errors='ignore')
    
    # Convert Yes and No columns to 1 and 0
    if yes_no_cols:
        # Standardize strings to lowercase and strip whitespaces
        for col in yes_no_cols:
            cleaned_df[col] = cleaned_df[col].astype(str).str.lower().str.strip()
            # Map values (unknown/other values will become NaN)
            cleaned_df[col] = cleaned_df[col].replace({'yes': 1, 'y': 1, 'no': 0, 'n': 0}).astype(int)
        
    return cleaned_df

#Split data into train and test data set, 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

#Identify feature data types
yes_no_feat = ['StockOptionLevel','OverTime']

num_feat = ['Age','DistanceFromHome_KM',
            'Education','Salary','OverTime',
            'YearsAtCompany','YearsInMostRecentRole',
            'YearsSinceLastPromotion','YearsWithCurrManager',
            'StockOptionLevel']

cat_feat = ['Gender','BusinessTravel','Department','State',
            'Ethnicity','EducationField','JobRole','MaritalStatus']

#Create cleaning transformer
cleaning_transformer = FunctionTransformer(
    clean_df, 
    kw_args={'yes_no_cols': yes_no_feat, 'cols_to_drop': drop_list}
)

#Create Preprocessing Sub-Pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Handle missing numeric entries
    ('scaler', StandardScaler()) # Scales features to mean=0, variance=1 for stable model convergence
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fill missing categories
    ('encoder', OneHotEncoder(handle_unknown='ignore'))   # Convert to numerical dummy variables
])

#Bundle Preprocessors Using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_feat),
        ('cat', categorical_transformer, cat_feat)
    ]
)

models = {
    'Random Forest': RandomForestClassifier(random_state=42),
}

#Initialize, Train, and Evaluate Models in a Loop
results = {}

for model_name, model_object in models.items():
    # Construct the final workflow pipeline dynamically
    full_pipeline = Pipeline(steps=[
        ('cleaning', cleaning_transformer),
        ('preprocessor', preprocessor),
        ('classifier', model_object)
    ])
    
    # Train the pipeline (fits transformers on train data only, then trains model)
    full_pipeline.fit(X_train, y_train)
    
    # Predict on unseen test data
    y_pred = full_pipeline.predict(X_test)
    
# Evaluate performance
    results[model_name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='binary', pos_label=1),
        'Recall': recall_score(y_test, y_pred, average='binary', pos_label=1),      
        'F1-Score': f1_score(y_test, y_pred, average='binary', pos_label=1)         
    }

#Display Performance Comparison
df_results = pd.DataFrame(results).T
df_results

,Accuracy,Precision,Recall,F1-Score
Random Forest,0.894531,0.956522,0.458333,0.619718


In [22]:
#Finding the feature importance scores

#Get Random Forest step our of pipeline
model = full_pipeline[-1]

#Find feature importance 
raw_importances = model.feature_importances_

# Isolate preprocessing step
preprocessor = full_pipeline.named_steps['preprocessor']

# Retrieving feature names from columns by running data through cleaning step
X_train_cleaned = full_pipeline.named_steps['cleaning'].transform(X_train)

# Then getting the dynamically generated feature names
feature_names = preprocessor.get_feature_names_out(input_features=X_train_cleaned.columns)

# Put feature importance into dataframe 
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': raw_importances
}).sort_values(by='Importance', ascending=False)

importance_df

,Feature,Importance
7,num__YearsSinceLastPromotion,0.157069
5,num__YearsAtCompany,0.156001
3,num__Salary,0.069053
0,num__Age,0.065223
8,num__YearsWithCurrManager,0.063785
6,num__YearsInMostRecentRole,0.060955
1,num__DistanceFromHome_KM,0.054032
4,num__OverTime,0.039810
2,num__Education,0.025694
9,num__StockOptionLevel,0.025271


In [13]:
import pickle

# Saving trained model to a file
with open('Attrition_model.pkl', 'wb') as file:
    pickle.dump(full_pipeline, file)

# Building Streamlit application

In [42]:
X_test.columns

Index(['EnvironmentSatisfaction', 'JobSatisfaction',
       'RelationshipSatisfaction', 'TrainingOpportunitiesWithinYear',
       'TrainingOpportunitiesTaken', 'WorkLifeBalance', 'SelfRating',
       'ManagerRating', 'Gender', 'Age', 'BusinessTravel', 'Department',
       'DistanceFromHome_KM', 'State', 'Ethnicity', 'Education',
       'EducationField', 'JobRole', 'MaritalStatus', 'Salary',
       'StockOptionLevel', 'OverTime', 'YearsAtCompany',
       'YearsInMostRecentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

In [95]:
#Function select user input
def user_input():

    st.sidebar.header("HR Data Input Features")
    
    EnvironmentSatisfaction = st.sidebar.number_input('EnvironmentSatisfaction', min_value=1, max_value=5, step=1, value=4)

    JobSatisfaction = st.sidebar.number_input('JobSatisfaction', min_value=1, max_value=5, step=1, value=4)

    RelationshipSatisfaction = st.sidebar.number_input('RelationshipSatisfaction', min_value=1, max_value=5, step=1, value=4)

    TrainingOpportunitiesWithinYear = st.sidebar.number_input('TrainingOpportunitiesWithinYear', min_value=1, max_value=5, step=1, value=4)

    TrainingOpportunitiesTaken = st.sidebar.number_input('TrainingOpportunitiesTaken', min_value=1, max_value=5, step=1, value=4)

    WorkLifeBalance = st.sidebar.number_input('WorkLifeBalance', min_value=1, max_value=5, step=1, value=4)

    SelfRating = st.sidebar.number_input('SelfRating', min_value=1, max_value=5, step=1, value=4)

    ManagerRating = st.sidebar.number_input('ManagerRating', min_value=1, max_value=5, step=1, value=4)
        
    Gender = st.sidebar.selectbox('Gender',['Female', 'Male', 'Non-Binary','Prefer Not To Say'])
    
    Age = st.sidebar.number_input('Age', min_value=0, max_value=65, step=1, value=30)
    
    BusinessTravel = st.sidebar.selectbox('BusinessTravel',['Some Travel', 'Frequent Traveller', 'Non-No Travel'])
    
    Department = st.sidebar.selectbox('Department',['Technology', 'Sales', 'Human Resources'])
    
    DistanceFromHome_KM = st.sidebar.number_input('DistanceFromHome_KM', min_value=0, max_value=100, step=1, value=30)
    
    State = st.sidebar.selectbox('State',['CA', 'NY', 'IL'])
    
    Ethnicity = st.sidebar.selectbox('Ethnicity',['White', 'Black or African American', 'Asian or Asian American ',
                                                 'American Indian or Alaska Native', 'Native Hawaiian ',
                                                 'Mixed or multiple ethnic groups', 'Other'])
    
    Education = st.sidebar.number_input('Education', min_value=0, max_value=5, step=1, value=3)
    
    EducationField = st.sidebar.selectbox('EducationField',['Computer Science', 'Marketing', 'Information Systems',
                                                 'Business Studies', 'Economics','Human Resources',
                                                 'Technical Degree', 'Other'])
    
    JobRole = st.sidebar.selectbox('JobRole',['Sales Executive','Software Engineer','Data Scientist',
                                   'Machine Learning Engineer','Senior Software Engineer',
                                  'Engineering Manager','Sales Representative',
                                   'Analytics Manager','Manager','HR Executive','Recruiter',
                                   'HR Business Partner','HR Manager'])
    
    MaritalStatus = st.sidebar.selectbox('MaritalStatus',['Single','Married','Divorced'])
    
    Salary = st.sidebar.number_input('Salary', min_value=0, max_value=1000000, step=1, value=100000)
    
    StockOptionLevel = st.sidebar.selectbox('OverTime',['Yes','No'])
    
    OverTime = st.sidebar.selectbox('OverTime',['Yes','No'])
    
    YearsAtCompany = st.sidebar.number_input('YearsAtCompany', min_value=0, max_value=65, step=1, value=30)
    
    YearsInMostRecentRole = st.sidebar.number_input('YearsInMostRecentRole', min_value=0, max_value=65, step=1, value=30)
    
    YearsSinceLastPromotion = st.sidebar.number_input('YearsSinceLastPromotion', min_value=0, max_value=65, step=1, value=30)
    
    YearsWithCurrManager = st.sidebar.number_input('YearsWithCurrManager', min_value=0, max_value=65, step=1, value=30)

    user_data = {
        'EnvironmentSatisfaction': EnvironmentSatisfaction,
        'JobSatisfaction': JobSatisfaction,
        'RelationshipSatisfaction': RelationshipSatisfaction,
        'TrainingOpportunitiesWithinYear': TrainingOpportunitiesWithinYear,
        'TrainingOpportunitiesTaken': TrainingOpportunitiesTaken,
        'WorkLifeBalance': WorkLifeBalance,
        'SelfRating': SelfRating,
        'ManagerRating': ManagerRating,
        'Gender': Gender,
        'Age': Age,
        'BusinessTravel': BusinessTravel,
        'Department': Department,
        'DistanceFromHome_KM': DistanceFromHome_KM,
        'State': State,
        'Ethnicity': Ethnicity,
        'Education': Education,
        'EducationField': EducationField,
        'JobRole': JobRole,
        'MaritalStatus': MaritalStatus,
        'Salary': Salary,
        'StockOptionLevel': StockOptionLevel,
        'OverTime': OverTime,
        'YearsAtCompany': YearsAtCompany,
        'YearsInMostRecentRole': YearsInMostRecentRole,
        'YearsSinceLastPromotion': YearsSinceLastPromotion,
        'YearsWithCurrManager': YearsWithCurrManager
    }
    features = pd.DataFrame([user_data])
    return features


In [81]:
user_data

,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating,Gender,Age,BusinessTravel,Department,DistanceFromHome_KM,State,Ethnicity,Education,EducationField,JobRole,MaritalStatus,Salary,StockOptionLevel,OverTime,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,4,4,4,4,4,4,4,4,Female,30,Some Travel,Technology,30,CA,White,3,Computer Science,Sales Executive,Single,100000,Yes,Yes,30,30,30,30


In [82]:
prediction = full_pipeline.predict(user_data)
prediction

/var/folders/p4/vxx4n3kn4q1grbl878lw90k80000gp/T/ipykernel_62195/4289377345.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_df[col] = cleaned_df[col].replace({'yes': 1, 'y': 1, 'no': 0, 'n': 0}).astype(int)
/var/folders/p4/vxx4n3kn4q1grbl878lw90k80000gp/T/ipykernel_62195/4289377345.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_df[col] = cleaned_df[col].replace({'yes': 1, 'y': 1, 'no': 0, 'n': 0}).astype(int)


array([0])

In [87]:
#Streamlit application

#Page Config
st.set_page_config(layout="wide")

#Load HR Attrition Model
with open('Attrition_model.pkl', 'rb') as file:
    model = pickle.load(file)

#HR data input sidebar
def user_input():

    st.sidebar.header("HR Data Input Features")
    
    EnvironmentSatisfaction = st.sidebar.number_input('EnvironmentSatisfaction', min_value=0, max_value=5, step=1, value=4)

    JobSatisfaction = st.sidebar.number_input('JobSatisfaction', min_value=0, max_value=5, step=1, value=4)

    RelationshipSatisfaction = st.sidebar.number_input('RelationshipSatisfaction', min_value=0, max_value=5, step=1, value=4)

    TrainingOpportunitiesWithinYear = st.sidebar.number_input('TrainingOpportunitiesWithinYear', min_value=0, max_value=5, step=1, value=4)

    TrainingOpportunitiesTaken = st.sidebar.number_input('TrainingOpportunitiesTaken', min_value=0, max_value=5, step=1, value=4)

    WorkLifeBalance = st.sidebar.number_input('WorkLifeBalance', min_value=0, max_value=5, step=1, value=4)

    SelfRating = st.sidebar.number_input('SelfRating', min_value=0, max_value=5, step=1, value=4)

    ManagerRating = st.sidebar.number_input('ManagerRating', min_value=0, max_value=5, step=1, value=4)
        
    Gender = st.sidebar.selectbox('Gender',['Female', 'Male', 'Non-Binary','Prefer Not To Say'])
    
    Age = st.sidebar.number_input('Age', min_value=0, max_value=65, step=1, value=30)
    
    BusinessTravel = st.sidebar.selectbox('BusinessTravel',['Some Travel', 'Frequent Traveller', 'Non-No Travel'])
    
    Department = st.sidebar.selectbox('Department',['Technology', 'Sales', 'Human Resources'])
    
    DistanceFromHome_KM = st.sidebar.number_input('DistanceFromHome_KM', min_value=0, max_value=100, step=1, value=30)
    
    State = st.sidebar.selectbox('State',['CA', 'NY', 'IL'])
    
    Ethnicity = st.sidebar.selectbox('Ethnicity',['White', 'Black or African American', 'Asian or Asian American ',
                                                 'American Indian or Alaska Native', 'Native Hawaiian ',
                                                 'Mixed or multiple ethnic groups', 'Other'])
    
    Education = st.sidebar.number_input('Education', min_value=0, max_value=5, step=1, value=3)
    
    EducationField = st.sidebar.selectbox('EducationField',['Computer Science', 'Information Systems', 'Asian or Asian American ',
                                                 'American Indian or Alaska Native', 'Native Hawaiian ',
                                                 'Mixed or multiple ethnic groups', 'Other'])
    
    JobRole = st.sidebar.selectbox('JobRole',['Sales Executive','Software Engineer','Data Scientist',
                                   'Machine Learning Engineer','Senior Software Engineer',
                                  'Engineering Manager','Sales Representative',
                                   'Analytics Manager','Manager','HR Executive','Recruiter',
                                   'HR Business Partner','HR Manager'])
    
    MaritalStatus = st.sidebar.selectbox('MaritalStatus',['Single','Married','Divorced'])
    
    Salary = st.sidebar.number_input('Salary', min_value=0, max_value=1000000, step=1, value=100000)
    
    StockOptionLevel = st.sidebar.selectbox('OverTime',['Yes','No'])
    
    OverTime = st.sidebar.selectbox('OverTime',['Yes','No'])
    
    YearsAtCompany = st.sidebar.number_input('YearsAtCompany', min_value=0, max_value=65, step=1, value=30)
    
    YearsInMostRecentRole = st.sidebar.number_input('YearsInMostRecentRole', min_value=0, max_value=65, step=1, value=30)
    
    YearsSinceLastPromotion = st.sidebar.number_input('YearsSinceLastPromotion', min_value=0, max_value=65, step=1, value=30)
    
    YearsWithCurrManager = st.sidebar.number_input('YearsWithCurrManager', min_value=0, max_value=65, step=1, value=30)

    user_data = {
        'EnvironmentSatisfaction': EnvironmentSatisfaction,
        'JobSatisfaction': JobSatisfaction,
        'RelationshipSatisfaction': RelationshipSatisfaction,
        'TrainingOpportunitiesWithinYear': TrainingOpportunitiesWithinYear,
        'TrainingOpportunitiesTaken': TrainingOpportunitiesTaken,
        'WorkLifeBalance': WorkLifeBalance,
        'SelfRating': SelfRating,
        'ManagerRating': ManagerRating,
        'Gender': Gender,
        'Age': Age,
        'BusinessTravel': BusinessTravel,
        'Department': Department,
        'DistanceFromHome_KM': DistanceFromHome_KM,
        'State': State,
        'Ethnicity': Ethnicity,
        'Education': Education,
        'EducationField': EducationField,
        'JobRole': JobRole,
        'MaritalStatus': MaritalStatus,
        'Salary': Salary,
        'StockOptionLevel': StockOptionLevel,
        'OverTime': OverTime,
        'YearsAtCompany': YearsAtCompany,
        'YearsInMostRecentRole': YearsInMostRecentRole,
        'YearsSinceLastPromotion': YearsSinceLastPromotion,
        'YearsWithCurrManager': YearsWithCurrManager
    }
    features = pd.DataFrame([user_data])
    return features

# Centered title
st.markdown("<h1 style='text-align: center;'>Attrition Prediction App</h1>", unsafe_allow_html=True)

st.header("Predict Employee Attrition")

#Get inputs from sidebar
user_data = user_input()

# Predict button
if st.button("Predict"):
        prediction = model.predict(user_data)
        st.subheader("Predicted Price")
        st.write(f'{prediction}')

# streamlit run HR_Attrition_predict.py


2026-07-22 14:36:54.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 14:36:54.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 14:36:54.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 14:36:54.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 14:36:54.727 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 14:36:54.728 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 14:36:54.728 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 14:36:54.729 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar